# Brief 5 charts

Produces the four charts for Brief 5: The Journey of an Instrument (CCT genealogy, 1997 to 2026).

Output PNGs at 300 DPI, editorial palette matching Briefs 3 and 4.

**Charts produced:**
- 01_transfer_adequacy.png (goes in Section 9)
- 02_coverage_share.png (goes in Section 9)
- 03_real_value_trajectory.png (goes in Section 9)
- 05_outcome_hierarchy.png (goes in Section 10A)

**Data sources:** CSVs in `data/`. Every value verified against a source in the Brief 5 source matrix.

Run all cells to regenerate all charts.

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import PercentFormatter

# Palette (matches Briefs 3 and 4)
INK      = '#14171a'
BURNT    = '#c05621'
AMBER    = '#d69e2e'
CREAM    = '#faf7f2'
PAPER    = '#ffffff'
RED      = '#a83232'
GREY_DK  = '#4a4a4a'
GREY_MD  = '#8a8a8a'
GREY_LT  = '#cfcfcf'

DPI  = 300
BASE = Path.cwd()
DATA = BASE / 'data'
OUT  = BASE

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.edgecolor': INK,
    'axes.labelcolor': INK,
    'xtick.color': INK,
    'ytick.color': INK,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.titlelocation': 'left',
    'figure.facecolor': PAPER,
    'axes.facecolor': PAPER,
    'savefig.facecolor': PAPER,
    'savefig.bbox': 'tight',
    'savefig.dpi': DPI,
})

def _title_block(ax, title, subtitle=None):
    ax.set_title(title, fontsize=14, color=INK, pad=18, loc='left')
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes,
                fontsize=10, color=GREY_DK, ha='left', va='bottom')

def _source_line(fig, text):
    fig.text(0.02, 0.005, text, fontsize=7.5, color=GREY_MD, ha='left')

## Chart 1. Transfer adequacy across programmes

Section 9. Shows the transfer size per programme in USD per household per month, sorted, with the 20 percent adequacy threshold from Bastagli et al. 2016 drawn as a vertical reference. Nigeria's NASSP-HUP shown twice: nominal at appraisal and real 2026 value.

In [ ]:
df = pd.read_csv(DATA / 'chart1_transfer_adequacy.csv')
df = df.sort_values('transfer_usd_month', ascending=True).reset_index(drop=True)
df['label'] = df['programme'] + ' (' + df['country'] + ')'

THRESHOLD_USD = 20
def _bar_color(row):
    if 'Nigeria' in row['country']:
        return RED
    return AMBER if row['transfer_usd_month'] >= THRESHOLD_USD else BURNT
colors = df.apply(_bar_color, axis=1).tolist()

fig, ax = plt.subplots(figsize=(10.5, 6.5))
ax.barh(df['label'], df['transfer_usd_month'], color=colors,
        edgecolor=INK, linewidth=0.6)
ax.axvline(THRESHOLD_USD, color=INK, linestyle='--', linewidth=1.2, alpha=0.75)
ax.text(THRESHOLD_USD + 1.5, len(df) - 0.4,
        'Adequacy threshold\n(20% of consumption,\nBastagli et al. 2016)',
        fontsize=8.5, color=INK, va='top')

for i, v in enumerate(df['transfer_usd_month']):
    ax.text(v + 1, i, f'USD {v:.1f}', va='center', fontsize=8.5, color=INK)

real_row = df.index[df['programme'] == 'NASSP-HUP real 2026']
if len(real_row):
    idx = real_row[0]
    ax.annotate(
        'Real value 2026, below USD 4',
        xy=(df.at[idx, 'transfer_usd_month'], idx),
        xytext=(35, idx - 0.05),
        fontsize=8.5, color=RED,
        arrowprops=dict(arrowstyle='->', color=RED, lw=0.8),
    )

ax.set_xlabel('USD per household per month', fontsize=10, color=INK)
ax.set_xlim(0, max(df['transfer_usd_month'].max() + 25, 80))
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=9)
ax.grid(axis='x', color=GREY_LT, linewidth=0.5, alpha=0.6)
ax.set_axisbelow(True)

_title_block(ax,
    'Transfer size across CCT programmes, USD per household per month',
    "Nigeria's NASSP-HUP sits below the adequacy threshold. Real value in 2026 is below USD 4.")

handles = [
    mpatches.Patch(color=AMBER, label='At or above adequacy threshold'),
    mpatches.Patch(color=BURNT, label='Below adequacy threshold'),
    mpatches.Patch(color=RED,   label='Nigeria (NASSP-HUP and COPE)'),
]
ax.legend(handles=handles, loc='lower right', frameon=False, fontsize=8.5)

_source_line(fig,
    'Source: Brief 5 country sources (Bastagli et al. 2016; F1 Fiszbein & Schady 2009; '
    'F4a IEG 2024; UNICEF Transfer Project). Values are nominal at last documented year.')
plt.savefig(OUT / '01_transfer_adequacy.png')
plt.show()

## Chart 2. Coverage as share of national population

Section 9. Compares coverage across programmes. Nigeria shown twice: achieved and target under NASSP-SU.

In [ ]:
df = pd.read_csv(DATA / 'chart2_coverage_share.csv')
df = df.sort_values('population_share_pct', ascending=True).reset_index(drop=True)
df['label'] = df['programme'] + ' (' + df['country'] + ')'

def _bar_color(row):
    if 'achieved' in row['programme']:
        return RED
    if 'target' in row['programme']:
        return AMBER
    return INK
colors = df.apply(_bar_color, axis=1).tolist()
edgecolors = [AMBER if 'target' in p else INK for p in df['programme']]

fig, ax = plt.subplots(figsize=(10.5, 6.5))
ax.barh(df['label'], df['population_share_pct'], color=colors,
        edgecolor=edgecolors, linewidth=1.0)

for i, v in enumerate(df['population_share_pct']):
    ax.text(v + 0.6, i, f'{v:.0f}%', va='center', fontsize=8.5, color=INK)

achieved_idx = df.index[df['programme'] == 'NASSP-HUP achieved']
if len(achieved_idx):
    idx = achieved_idx[0]
    ax.annotate(
        '1.8M households, below 2%',
        xy=(df.at[idx, 'population_share_pct'], idx),
        xytext=(15, idx - 0.05),
        fontsize=8.5, color=RED,
        arrowprops=dict(arrowstyle='->', color=RED, lw=0.8),
    )

ax.set_xlabel('Share of national population covered (%)', fontsize=10, color=INK)
ax.xaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.set_xlim(0, max(df['population_share_pct'].max() + 10, 70))
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=9)
ax.grid(axis='x', color=GREY_LT, linewidth=0.5, alpha=0.6)
ax.set_axisbelow(True)

_title_block(ax,
    'Cash transfer coverage as share of national population',
    "Nigeria's achieved coverage sits with Ghana and Kenya. The scale-up target would move it toward the Latin American range.")

handles = [
    mpatches.Patch(color=INK,   label='Other programmes'),
    mpatches.Patch(color=RED,   label='Nigeria NASSP-HUP achieved'),
    mpatches.Patch(color=AMBER, label='Nigeria NASSP-SU target (aspirational)'),
]
ax.legend(handles=handles, loc='lower right', frameon=False, fontsize=8.5)

_source_line(fig,
    'Source: Brief 5 country sources; F4a IEG 2024; F4b NASSP-SU PAD 2024. '
    'JSY shown as annual flow (one-time payment per delivery).')
plt.savefig(OUT / '02_coverage_share.png')
plt.show()

## Chart 3. Real-value trajectory of cash transfers, 2016 to 2026

Section 9. Nigeria NASSP-HUP shown against Kenya CT-OVC, Brazil Bolsa, Ghana LEAP, and Mexico Prospera. All values in USD per household per month, deflated to 2016 base.

In [ ]:
df = pd.read_csv(DATA / 'chart3_real_value_trajectory.csv')

fig, ax = plt.subplots(figsize=(10.5, 6.5))

ax.plot(df['year'], df['ct_ovc_kenya'],
        color=GREY_DK, linewidth=1.6, linestyle='-',
        label='Kenya CT-OVC (indexed)')
ax.plot(df['year'], df['bolsa_brazil'],
        color=BURNT, linewidth=1.6, linestyle=':',
        label='Brazil Bolsa Familia')
ax.plot(df['year'], df['leap_ghana'],
        color=GREY_MD, linewidth=1.6, linestyle='--',
        label='Ghana LEAP')
ax.plot(df['year'], df['prospera_mexico'],
        color=AMBER, linewidth=1.6, linestyle='-.',
        label='Mexico Prospera (replaced 2019)')
ax.plot(df['year'], df['nassp_hup_nigeria'],
        color=RED, linewidth=3.2, marker='o', markersize=5,
        label='Nigeria NASSP-HUP')

ax.axhline(20, color=INK, linestyle='--', linewidth=1.0, alpha=0.6)
ax.text(2016, 21, 'Adequacy threshold (USD 20)',
        fontsize=8.5, color=INK, va='bottom')

ax.annotate('Nigeria set NGN 5,000 in 2016.\nNo indexation clause.',
            xy=(2016, 12.17), xytext=(2016.4, 30),
            fontsize=8.5, color=RED,
            arrowprops=dict(arrowstyle='->', color=RED, lw=0.8))
ax.annotate('January 2024 suspension, 12.5 months',
            xy=(2024, 4.2), xytext=(2020.5, 12),
            fontsize=8.5, color=RED,
            arrowprops=dict(arrowstyle='->', color=RED, lw=0.8))
ax.annotate('Lula restructuring (Bolsa)',
            xy=(2023, 42), xytext=(2019.5, 48),
            fontsize=8.5, color=BURNT,
            arrowprops=dict(arrowstyle='->', color=BURNT, lw=0.8))

ax.set_xlabel('Year', fontsize=10, color=INK)
ax.set_ylabel('Real transfer value, USD per household per month\n(2016 base year)',
              fontsize=10, color=INK)
ax.set_xlim(2015.5, 2026.5)
ax.set_ylim(0, 60)
ax.grid(axis='both', color=GREY_LT, linewidth=0.5, alpha=0.6)
ax.set_axisbelow(True)

_title_block(ax,
    'Real-value trajectory of cash transfers, 2016 to 2026',
    "Nigeria's NASSP-HUP shows a monotone decline unmatched by any comparator.")

ax.legend(loc='upper right', frameon=False, fontsize=8.5)
_source_line(fig,
    'Source: Brief 5 country sources; NBS Food CPI; F4a IEG 2024; F4b NASSP-SU PAD 2024. '
    'Values estimated where indexation not documented.')
plt.savefig(OUT / '03_real_value_trajectory.png')
plt.show()

## Chart 5. CCT effect hierarchy by outcome domain

Section 10A. The four outcome domains ranked by evidence strength, with a Nigeria indicator column showing which domains NASSP-HUP formally targets.

In [ ]:
df = pd.read_csv(DATA / 'chart5_outcome_hierarchy.csv')
df = df.sort_values('evidence_strength_score', ascending=True).reset_index(drop=True)

color_map = {4: AMBER, 3: AMBER, 2: BURNT, 1: RED}
colors = [color_map[s] for s in df['evidence_strength_score']]

fig, ax = plt.subplots(figsize=(12.5, 6.2))
ax.barh(df['outcome_domain'], df['evidence_strength_score'],
        color=colors, edgecolor=INK, linewidth=0.6, height=0.55)

for i, row in df.iterrows():
    eff = row['effect_size_range']
    if len(eff) > 68:
        eff = eff[:65] + '...'
    ax.text(row['evidence_strength_score'] + 0.08, i, '  ' + eff,
            va='center', fontsize=9, color=INK)

NIGERIA_COL_X = 8.9
for i, row in df.iterrows():
    v = row['nigeria_targets_this']
    if v == 'yes':
        ax.text(NIGERIA_COL_X, i, '★ Nigeria targets this domain',
                va='center', fontsize=9, color=RED, ha='left', weight='bold')
    elif v == 'partial':
        ax.text(NIGERIA_COL_X, i, '◐ Nigeria targets partially',
                va='center', fontsize=9, color=BURNT, ha='left')
    else:
        ax.text(NIGERIA_COL_X, i, '○ Not targeted by Nigeria',
                va='center', fontsize=9, color=GREY_MD, ha='left')

ax.set_xlim(0, 14.5)
ax.set_xticks([])
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=10.5)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_color(INK)

_title_block(ax,
    'CCT effect hierarchy by outcome domain',
    "Nigeria's NASSP-HUP targets education access and health utilisation, the two domains where the instrument works most.")

handles = [
    mpatches.Patch(color=AMBER, label='Strong evidence base'),
    mpatches.Patch(color=BURNT, label='Moderate evidence base'),
    mpatches.Patch(color=RED,   label='Weakest evidence base'),
]
ax.legend(handles=handles, loc='lower right', frameon=False, fontsize=8.5)

_source_line(fig,
    'Source: Brief 5 Section 10A synthesis of Bastagli et al. 2016, '
    'Baird et al. 2011, and country-level evidence tabulated in the source matrix.')
plt.savefig(OUT / '05_outcome_hierarchy.png')
plt.show()

## Done

All four charts saved as PNG in this directory. If you edit any CSV in `data/`, re-run the corresponding cell.